## LLM experiments: Dictator Game & Ultimatum Game

This notebook:
- queries OpenRouter models for Dictator/Ultimatum game prompts
- varies social distance, money amount
- parses responses in the template: `I share [amount] dollars`
- stores raw + parsed results and makes simple plots

### Setup
- Set `OPENROUTER_API_KEY` in your environment.
- Optional: set `OPENROUTER_SITE_URL` and `OPENROUTER_APP_NAME`.

### Smoke test
By default, the notebook runs only **2 random** combinations of:
`(model, social_distance, shared_money)`.
Disable smoke test to run the full grid.

### Repetitions
Set `N_REPEATS` to run multiple independent calls per condition.
Plots use the average over repetitions.



In [ ]:
import os
import re
import json
import time
import random
from datetime import datetime, timezone

import requests
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt


pd.set_option("display.max_colwidth", 120)



In [ ]:
# -----------------
# Experiment config
# -----------------

BASE_DIR = "/home/demid/Научная работа с Дагаевым/exp"


def _load_dotenv(path: str) -> None:
    if not os.path.exists(path):
        return

    with open(path, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue

            key, val = line.split("=", 1)
            key = key.strip()
            val = val.strip().strip('"').strip("'")
            if key and key not in os.environ:
                os.environ[key] = val


_load_dotenv(os.path.join(BASE_DIR, ".env"))

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "").strip()
OPENROUTER_SITE_URL = os.getenv("OPENROUTER_SITE_URL", "")
OPENROUTER_APP_NAME = os.getenv("OPENROUTER_APP_NAME", "")

if not OPENROUTER_API_KEY:
    raise RuntimeError(
        "Missing OPENROUTER_API_KEY env var (or .env). "
        "Export it and re-run this notebook."
    )

OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"

MODELS = [
    "deepseek/deepseek-v3.2",
    "x-ai/grok-4.1-fast",
    "anthropic/claude-sonnet-4.5",
]

SOCIAL_DISTANCES = [2, 20, 100]

MONEY_AMOUNTS = [10, 3000, 250_000]

# Full run
SMOKE_TEST = False
SMOKE_N = 2
SMOKE_SEED = 123

N_REPEATS = 3
REQUEST_SLEEP_S = 0.2

OUT_DIR = os.path.join(BASE_DIR, "out")
os.makedirs(OUT_DIR, exist_ok=True)

RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")



In [ ]:
# -----------------
# Prompts (verbatim)
# -----------------

DICTATOR_PROMPT_TMPL = (
    "You will be asked to make a decision regarding how much\n"
    "money you might offer another person under a situation\n"
    "that will be explained shortly. There are no correct or incorrect\n"
    "answers, and the money is hypothetical – that is, no one will "
    "receive\n"
    "the actual money. Nonetheless, we want you to make your decision\n"
    "as if the amount and situation were real.\n"
    "Before describing the situation, we want you to imagine that you\n"
    "have made a list of the 100 people closest to you in the world,\n"
    "ranging from your dearest friend or relative at position #1 to a\n"
    "mere acquaintance at #100. The person at number one would be\n"
    "someone you know well and is your closest friend or relative. The\n"
    "person at #100 might be someone you recognize and encounter but\n"
    "perhaps you may not even know their name. You do not have to\n"
    "create the list – just imagine that you have done so.\n"
    "Imagine you have been given the amount of money: {m}\n"
    "You are to divide the amount of money between\n"
    "yourself and another person who is in place {x} at that list.\n"
    "You are free to give as much or as little\n"
    "of the amount of money as you wish, and you will receive what is\n"
    "left. Please respond with the amount you wish to offer the other\n"
    "person. Follow the exact answer template: \"I share "
    "[decided amount] dollars\"."
)

ULTIMATUM_PROMPT_TMPL = (
    "You will be asked to make a decision regarding how much\n"
    "money you might offer another person under a situation\n"
    "that will be explained shortly. There are no correct or incorrect\n"
    "answers, and the money is hypothetical – that is, no one will "
    "receive\n"
    "the actual money. Nonetheless, we want you to make your decision\n"
    "as if the amount and situation were real.\n"
    "Before describing the situation, we want you to imagine that you\n"
    "have made a list of the 100 people closest to you in the world,\n"
    "ranging from your dearest friend or relative at position #1 to a\n"
    "mere acquaintance at #100. The person at number one would be\n"
    "someone you know well and is your closest friend or relative. The\n"
    "person at #100 might be someone you recognize and encounter but\n"
    "perhaps you may not even know their name. You do not have to\n"
    "create the list – just imagine that you have done so.\n"
    "Imagine you have been given the amount of money: {m}\n"
    "You are to divide the amount of money between\n"
    "yourself and another person who is in place {x} at that list.\n"
    "You are free to give as much or as little\n"
    "of the amount of money as you wish, and you will receive what is\n"
    "left, but only if the other person accepts your offer. If the other\n"
    "person rejects your offer, however, then both of you will receive\n"
    "nothing. Please respond with the amount you wish to offer the "
    "other\n"
    "person. Follow the exact answer template: \"I share "
    "[decided amount] dollars\"."
)


def build_prompt(game: str, x: int, m: int | None) -> str:
    if m is None:
        raise ValueError("Prompt requires m")

    if game == "dictator":
        return DICTATOR_PROMPT_TMPL.format(m=f"${m}", x=x)
    if game == "ultimatum":
        return ULTIMATUM_PROMPT_TMPL.format(m=f"${m}", x=x)
    raise ValueError(f"Unknown game: {game}")



In [ ]:
# -----------------
# OpenRouter client
# -----------------


def _openrouter_headers() -> dict[str, str]:
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    if OPENROUTER_SITE_URL:
        headers["HTTP-Referer"] = OPENROUTER_SITE_URL
    if OPENROUTER_APP_NAME:
        headers["X-Title"] = OPENROUTER_APP_NAME
    return headers


def _content_to_text(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


def openrouter_chat(
    model: str,
    user_text: str,
    timeout_s: int = 60,
    max_retries: int = 3,
) -> tuple[str, dict]:
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": user_text}],
        # Keep temperature at default: do not send `temperature`.
    }

    last_exc: Exception | None = None
    for attempt in range(max_retries):
        try:
            t0 = time.time()
            resp = requests.post(
                OPENROUTER_API_URL,
                headers=_openrouter_headers(),
                json=payload,
                timeout=timeout_s,
            )
            dt = time.time() - t0

            if resp.status_code >= 500:
                time.sleep(1.5 * (attempt + 1))
                continue

            resp.raise_for_status()
            data = resp.json()
            content = data["choices"][0]["message"]["content"]

            meta = {"data": data, "latency_s": dt}
            return _content_to_text(content), meta
        except Exception as exc:
            last_exc = exc
            time.sleep(1.5 * (attempt + 1))

    raise RuntimeError("OpenRouter request failed") from last_exc



In [ ]:
# -----------------
# Parsing utilities
# -----------------

_SHARE_RE = re.compile(
    (
        r"\bi\s*share\s*([$€£]?\s*[0-9][0-9,]*"
        r"(?:\.[0-9]+)?)\s*dollars\b"
    ),
    re.IGNORECASE,
)


def parse_share_amount(text: str) -> tuple[float | None, str | None]:
    if not text or not text.strip():
        return None, "empty"

    m = _SHARE_RE.search(text)
    if not m:
        return None, "no_match"

    raw = m.group(1)
    raw = raw.replace(",", "")
    raw = raw.replace("$", "").replace("€", "").replace("£", "")
    raw = raw.strip()

    try:
        return float(raw), None
    except ValueError:
        return None, "parse_error"



In [ ]:
# -----------------
# Experiment runner
# -----------------


def sample_combos() -> list[tuple[str, int, int]]:
    all_combos = [
        (model, x, m)
        for model in MODELS
        for x in SOCIAL_DISTANCES
        for m in MONEY_AMOUNTS
    ]

    if not SMOKE_TEST:
        return all_combos

    rng = random.Random(SMOKE_SEED)
    if SMOKE_N > len(all_combos):
        raise ValueError("SMOKE_N exceeds the number of combinations")
    return rng.sample(all_combos, k=SMOKE_N)


def _summarize(df_raw: pd.DataFrame) -> pd.DataFrame:
    grp = ["model", "game", "social_distance", "money"]

    def _agg(g: pd.DataFrame) -> pd.Series:
        share = g["share"].astype(float)
        share_pct = g["share_pct"].astype(float)

        n = int(len(g))
        n_valid = int(np.isfinite(share).sum())
        n_parse_err = int(g["parse_error"].notna().sum())

        out = {
            "n": n,
            "n_valid": n_valid,
            "parse_error_rate": n_parse_err / n,
            "share_mean": float(np.nanmean(share)),
            "share_std": float(np.nanstd(share, ddof=1))
            if n_valid >= 2
            else np.nan,
            "share_pct_mean": float(np.nanmean(share_pct)),
            "share_pct_std": float(np.nanstd(share_pct, ddof=1))
            if n_valid >= 2
            else np.nan,
        }
        return pd.Series(out)

    df = df_raw.groupby(grp, dropna=False).apply(_agg).reset_index()
    return df


def run_experiments() -> tuple[pd.DataFrame, pd.DataFrame]:
    combos = sample_combos()

    jobs: list[dict] = []

    for model, x, m in combos:
        jobs.append({"model": model, "game": "dictator", "x": x, "m": m})
        jobs.append({"model": model, "game": "ultimatum", "x": x, "m": m})

    rows: list[dict] = []
    job_total = len(jobs)

    for job_i, job in enumerate(jobs, start=1):
        model = job["model"]
        game = job["game"]
        x = int(job["x"])
        m = int(job["m"]) if job["m"] is not None else None

        prompt = build_prompt(game=game, x=x, m=m)

        for rep in range(1, N_REPEATS + 1):
            print(
                f"[{job_i}/{job_total}]",
                model,
                "|",
                game,
                "|",
                f"x={x}",
                "|",
                f"m={m}",
                "|",
                f"rep={rep}/{N_REPEATS}",
            )

            created_utc = datetime.now(timezone.utc).isoformat()
            try:
                text, meta = openrouter_chat(model=model, user_text=prompt)
                share, err = parse_share_amount(text)
                latency_s = float(meta.get("latency_s", np.nan))
                openrouter = meta.get("data", {})
            except Exception as exc:
                text = ""
                share = np.nan
                err = f"request_error: {type(exc).__name__}"
                latency_s = np.nan
                openrouter = {}

            rows.append(
                {
                    "run_tag": RUN_TAG,
                    "created_utc": created_utc,
                    "model": model,
                    "game": game,
                    "social_distance": x,
                    "money": m,
                    "rep": rep,
                    "raw_response": text,
                    "share": share,
                    "parse_error": err,
                    "latency_s": latency_s,
                    "openrouter": openrouter,
                }
            )

            time.sleep(REQUEST_SLEEP_S)

    df_raw = pd.DataFrame(rows)
    df_raw["share_pct"] = df_raw["share"] / df_raw["money"].astype(float)

    df_avg = _summarize(df_raw)

    out_jsonl = os.path.join(OUT_DIR, f"results_{RUN_TAG}.jsonl")
    out_csv = os.path.join(OUT_DIR, f"results_{RUN_TAG}.csv")
    out_avg_csv = os.path.join(OUT_DIR, f"results_{RUN_TAG}_avg.csv")

    with open(out_jsonl, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    df_raw.drop(columns=["openrouter"]).to_csv(out_csv, index=False)
    df_avg.to_csv(out_avg_csv, index=False)

    print(f"Saved: {out_jsonl}")
    print(f"Saved: {out_csv}")
    print(f"Saved: {out_avg_csv}")

    return df_raw, df_avg



In [ ]:
df_raw, df_avg = run_experiments()

df_avg.sort_values(
    ["model", "game", "money", "social_distance"]
).reset_index(drop=True)



In [ ]:
# -----------------
# Plots (saved to out/)
# -----------------


def _model_slug(model: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", model).strip("_")


def _savefig(fig: plt.Figure, name: str) -> str:
    out = os.path.join(OUT_DIR, name)
    fig.savefig(out, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return out


def _plot_dictator_by_money(df_in: pd.DataFrame, model: str) -> None:
    d = df_in[(df_in["game"] == "dictator") & (df_in["model"] == model)]
    d = d.copy()
    if d.empty:
        return

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for m in sorted(d["money"].unique()):
        dm = d[d["money"] == m].sort_values(["social_distance"])
        ax.plot(
            dm["social_distance"],
            dm["share_pct_mean"],
            marker="o",
            label=f"m=${int(m)}",
        )

    ax.set_title(
        "dictator (avg over reps): share/money vs distance\n" + model
    )
    ax.set_xlabel("social distance (x)")
    ax.set_ylabel("mean share / money")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

    slug = _model_slug(model)
    _savefig(fig, f"plot_{RUN_TAG}_{slug}_dictator_by_money_avg.png")


def _plot_ultimatum_by_money(df_in: pd.DataFrame, model: str) -> None:
    d = df_in[(df_in["game"] == "ultimatum") & (df_in["model"] == model)]
    d = d.copy()
    if d.empty:
        return

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for m in sorted(d["money"].unique()):
        dm = d[d["money"] == m].sort_values(["social_distance"])
        ax.plot(
            dm["social_distance"],
            dm["share_pct_mean"],
            marker="o",
            label=f"m=${int(m)}",
        )

    ax.set_title(
        "ultimatum (avg over reps): share/money vs distance\n" + model
    )
    ax.set_xlabel("social distance (x)")
    ax.set_ylabel("mean share / money")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

    slug = _model_slug(model)
    _savefig(fig, f"plot_{RUN_TAG}_{slug}_ultimatum_by_money_avg.png")


for model in sorted(df_avg["model"].unique()):
    _plot_dictator_by_money(df_avg, model)
    _plot_ultimatum_by_money(df_avg, model)

